# LangChain Tutorial
### Build a Media Intelligence Pipeline — from raw articles to sentiment analysis

---

## The story

A client wants to monitor their brand across news articles, press releases and online media.
Every morning they need:
- **Sentiment scores** for each piece of coverage
- **Key entities** — who said what, where, when
- **Answers to ad-hoc questions** about any article in the archive
- A **conversational assistant** that remembers context across questions

We will build exactly that — step by step — using LangChain and Google Gemini.

---

## What is an LLM?

A Large Language Model is a neural network trained on billions of text documents.
It learns statistical patterns across language and can generate, classify, translate and
reason about text. It is stateless — it receives a prompt and returns a completion.
It knows nothing about your client's documents unless you inject that context into the prompt.

That is the core problem this notebook solves.

## What is LangChain?

LangChain is a framework that provides one consistent interface across:

| Layer | What it gives you |
|---|---|
| **Model wrappers** | One interface for Gemini, GPT, Claude, local models |
| **Prompt templates** | Reusable, parameterised prompts |
| **Document loaders** | Load PDFs, web pages, CSVs, plain text |
| **Text splitters** | Break long docs into chunks |
| **Embeddings + vector stores** | Semantic search over your documents |
| **Chains (LCEL)** | Pipe components together with `|` |
| **Memory** | Conversation history |
| **Agents + tools** | LLM that decides which function to call |

## The full pipeline

```
  [Media articles / PDFs / web pages]
            |
            |  Part 2 — Document Loaders
            v
      List[Document]   (page_content + metadata)
            |
            |  Part 3 — Text Splitter
            v
      List[Chunk]      (short, overlapping passages)
            |
            |  Part 4 — Gemini Embeddings  (FREE)
            v
      FAISS index      (vectors on disk)
            |
            |  Part 5 — RAG chain
            v
      Grounded answers + source citations
            |
            |  Part 6 — Sentiment & Entity pipeline
            v
      Structured JSON report per article
            |
            |  Part 7 — ReAct Agent
            v
      Conversational assistant with memory
```

---
## Part 0 — Install & API Key

Gemini key https://aistudio.google.com/app/apikey

Deepseek key https://platform.deepseek.com/usage

OpenAI key https://platform.openai.com/home


In [1]:
!pip install langchain_core langchain_community langchain-google-genai langchain_classic langchain_huggingface langchain_openai langchain_text_splitters faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
GOOGLE_API_KEY = "AQ...."

In [3]:
DEEPSEEK_API_KEY = "sk..."

In [4]:
from openai import OpenAI

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

models = client.models.list()
for model in models.data:
    print(f"Model ID: {model.id}")

Model ID: deepseek-v4-flash
Model ID: deepseek-v4-pro


In [5]:
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [6]:
# print("Models supporting Embeddings:")
# print("-" * 30)

# for m in genai.list_models():
#     if 'embedContent' in m.supported_generation_methods:
#         print(m.name)

# for m in genai.list_models():
#     if 'embedContent' in m.supported_generation_methods:
#         print(f"Model Name: {m.name}")
#         print(f"Description: {m.description}\n")

In [7]:
# Run once, then restart the kernel if prompted
# !pip install langchain langchain-core langchain-community langchain-google-genai \
#              langchain-text-splitters faiss-cpu pypdf beautifulsoup4 requests

# ── API Key — paste yours here ───────────────────────────────────────────────


# Model selection
# Gemini options:        gemini-1.5-flash (cheap, fast) | gemini-1.5-pro (more capable) | gemini-2.0-flash (newest)
# OpenAI equivalents:    gpt-4o-mini (≈ flash)          | gpt-4o (≈ pro)               | gpt-4.5 (most capable, expensive)
# Anthropic equivalents: claude-haiku-4-5 (≈ flash)     | claude-sonnet-4-5 (≈ pro)    | claude-opus-4-5 (most capable)
# All three work identically below — just swap the class and key.

CHAT_MODEL  = "deepseek-v4-flash"
EMBED_MODEL = "all-MiniLM-L6-v2"

print("Keys configured")

Keys configured


In [8]:
# Sample media articles — self-contained so no files are needed
# In a real project: TextLoader("file.txt") / PyPDFLoader("report.pdf") / WebBaseLoader(url)

SAMPLE_ARTICLES = {
    "techcrunch_ai_funding.txt": """
AI Startup NeuralBridge Raises $120M Series C to Expand Enterprise Platform
TechCrunch | January 14, 2025

NeuralBridge, the San Francisco-based AI infrastructure startup, announced today
that it has closed a $120 million Series C funding round led by Sequoia Capital,
with participation from Google Ventures and Andreessen Horowitz.

The company, founded in 2021 by former DeepMind researchers Dr Sarah Chen and
Marcus Webb, has grown its enterprise customer base from 40 to over 300 companies
in the past 18 months. Notable clients include JPMorgan Chase, Siemens and
the UK National Health Service.

CEO Dr Chen said the new capital would be used to expand the engineering team
from 180 to 400 employees and accelerate product development in the EU market.
The company plans to open offices in London and Berlin by Q3 2025.

The funding values NeuralBridge at approximately $850 million.
The AI infrastructure market is projected to reach $200 billion by 2027, per Gartner.
""",

    "reuters_ai_regulation.txt": """
EU Regulators Tighten AI Oversight Rules, Threatening Billions in Fines
Reuters | January 16, 2025

European Union regulators announced sweeping new enforcement guidelines for the
EU AI Act on Thursday, warning that violations could result in fines of up to
7 percent of global annual revenue for companies deploying high-risk AI systems.

The guidelines, issued by the European AI Office in Brussels, clarify that biometric
surveillance systems, AI used in hiring decisions, and credit-scoring algorithms
will all be classified as high-risk and subject to mandatory third-party audits.

Industry groups expressed concern that the compliance burden could disadvantage
European AI firms against US and Chinese competitors. TechEurope, a lobby group
representing over 600 technology companies, called the guidelines "disproportionate
and potentially damaging to European innovation."

The European Commission defended the rules, stating that consumer protection and
fundamental rights must take precedence. Commissioner Helena Vasquez said the EU
is committed to AI that is trustworthy, transparent and accountable.
""",

    "guardian_ai_jobs.txt": """
One in Four Jobs Could Be Automated by AI Within a Decade, IMF Warns
The Guardian | January 18, 2025

The International Monetary Fund warned on Friday that approximately 40 percent
of jobs globally are exposed to AI automation, with advanced economies facing
the highest levels of disruption — up to 60 percent in countries like the United
States, Germany and Japan.

The IMF's report distinguishes between jobs that will be replaced by AI and those
that will be augmented. Routine cognitive tasks — data entry, basic analysis,
customer service — face the greatest risk. Creative, social and physical roles
are considered more resilient.

The fund called on governments to invest urgently in retraining programmes and
to reform tax systems that currently favour capital over labour. IMF Managing
Director Kristalina Georgieva said the transition could either increase inequality
dramatically or create a more prosperous society depending on policy choices.
""",

    "bloomberg_nvidia.txt": """
NVIDIA Posts Record $35B Quarter as AI Chip Demand Shows No Sign of Slowing
Bloomberg | January 20, 2025

NVIDIA reported quarterly revenue of $35.1 billion on Wednesday, smashing analyst
expectations of $33 billion and representing a 122 percent year-over-year increase.
The results were driven almost entirely by data centre sales, which reached $30.8
billion as hyperscalers and enterprise customers continued to build out AI infrastructure.

CEO Jensen Huang told analysts that demand for the company's Blackwell GPU architecture
continues to exceed supply. Shares rose 8 percent in after-hours trading.

Analysts at Morgan Stanley raised their price target for NVIDIA to $900 per share.
However, some investors have raised concerns about concentration risk, noting that
Microsoft, Google, Amazon and Meta together account for roughly 40 percent of revenue.
""",

    "ft_ai_ethics.txt": """
Big Tech AI Ethics Commitments Under Scrutiny After Chatbot Failures
Financial Times | January 22, 2025

A wave of high-profile AI chatbot failures has renewed pressure on technology
companies to back up their published ethics principles with meaningful accountability.

In separate incidents, an AI customer service system at a major airline fabricated
refund policies, a legal research tool hallucinated case law cited in court documents,
and a healthcare chatbot provided dangerously incorrect dosage information to patients.

Alex Rivera, a researcher at the Oxford Internet Institute, said the incidents
illustrate a gap between AI safety principles that companies publish and the actual
testing and monitoring conducted before deployment. "Ethics washing is becoming a
serious problem," Rivera told the FT.

Google, Microsoft and OpenAI all declined to comment on the specific incidents
but pointed to their published responsible AI frameworks.
"""
}

print(f"Loaded {len(SAMPLE_ARTICLES)} sample media articles")
for name in SAMPLE_ARTICLES:
    print(f"  * {name}")

Loaded 5 sample media articles
  * techcrunch_ai_funding.txt
  * reuters_ai_regulation.txt
  * guardian_ai_jobs.txt
  * bloomberg_nvidia.txt
  * ft_ai_ethics.txt


---
## Part 1 — LangChain Core: LLM, Prompts, Chains

Three primitives that everything else is built from:

```
  PromptTemplate  ->  fill {variables}  ->  formatted string
  ChatModel       ->  send prompt       ->  AIMessage
  OutputParser    ->  extract content   ->  Python object

  Chain = PromptTemplate | ChatModel | OutputParser
  (LCEL pipe — like Unix pipes, each output feeds the next input)
```

In [9]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model=CHAT_MODEL,
#     google_api_key=GOOGLE_API_KEY,
#     temperature=0,                        # 0 = deterministic; increase for creativity
#     convert_system_message_to_human=True, # required for Gemini; not needed for OpenAI/Claude
# )

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=CHAT_MODEL,
    openai_api_key=DEEPSEEK_API_KEY,      # Your DeepSeek API Key
    openai_api_base="https://api.deepseek.com", # Points to DeepSeek servers
    temperature=0,
    # convert_system_message_to_human is NOT needed for DeepSeek
)

response = llm.invoke("What is LangChain in one sentence?")
print(response.content)


# Using OpenAI:    from langchain_openai import ChatOpenAI; llm = ChatOpenAI(model="gpt-4o", api_key=...)
# Using Anthropic: from langchain_anthropic import ChatAnthropic; llm = ChatAnthropic(model="claude-opus-4-5", api_key=...)

LangChain is a framework for building applications powered by large language models, enabling the chaining of prompts, data sources, and model interactions into modular pipelines.


In [10]:
# ── PromptTemplate — reusable prompts with {variables} ────────────────────────
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Single-variable
simple = PromptTemplate.from_template(
    "Summarise this headline for an executive, noting the key actor:\n\n{headline}"
)

# System + human (chat style)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a media analyst specialising in {industry}. Be concise."),
    ("human", "{question}"),
])

# .invoke() fills the placeholders and returns a formatted prompt
filled = chat_prompt.invoke({
    "industry": "AI and technology",
    "question": "What are the three biggest risks for AI companies in 2026?"
})
print(filled.messages)

[SystemMessage(content='You are a media analyst specialising in AI and technology. Be concise.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What are the three biggest risks for AI companies in 2026?', additional_kwargs={}, response_metadata={})]


In [11]:
# ── Output parsers ────────────────────────────────────────────────────────────
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# StrOutputParser: AIMessage -> plain string (most common)
# JsonOutputParser: AIMessage -> Python dict
# CommaSeparatedListOutputParser: "a, b, c" -> ["a", "b", "c"]

list_parser = CommaSeparatedListOutputParser()
print("List parser format hint:", list_parser.get_format_instructions())

List parser format hint: Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [12]:
# ── LCEL chain with | pipe ─────────────────────────────────────────────────────
# chain = prompt | llm | parser
# Each component's output becomes the next component's input.

summarise_chain = (
    PromptTemplate.from_template(
        "Summarise this news article in 2 sentences for an executive briefing:\n\n{article}"
    )
    | llm
    | StrOutputParser()
)

result = summarise_chain.invoke({"article": SAMPLE_ARTICLES["techcrunch_ai_funding.txt"]})
print(result)

NeuralBridge raised $120M in Series C funding led by Sequoia Capital, with participation from Google Ventures and Andreessen Horowitz, at an $850M valuation. The AI infrastructure startup has scaled its enterprise customer base from 40 to over 300, including JPMorgan Chase and the UK NHS, and plans to use the capital to expand its engineering team and open European offices in London and Berlin by Q3 2025.


In [13]:
# ── .batch() — run a chain over a list in parallel ─────────────────────────────
# Much faster than calling .invoke() in a loop

batch_inputs = [{"article": text} for text in SAMPLE_ARTICLES.values()]
summaries = summarise_chain.batch(batch_inputs)

for name, summary in zip(SAMPLE_ARTICLES.keys(), summaries):
    print(f"[{name}]")
    print(f"  {summary}")
    print()

[techcrunch_ai_funding.txt]
  NeuralBridge raised a $120 million Series C led by Sequoia Capital at an $850 million valuation, underscoring strong investor confidence in the AI infrastructure market. The startup has grown its enterprise client base from 40 to over 300 in 18 months—including JPMorgan and the UK’s NHS—and will use the funds to double its engineering team and expand into Europe with new offices in London and Berlin.

[reuters_ai_regulation.txt]
  EU regulators have issued new enforcement guidelines for the EU AI Act, classifying biometric surveillance, hiring algorithms, and credit-scoring as high-risk systems subject to mandatory audits and fines of up to 7% of global annual revenue. Industry groups warn the compliance burden could disadvantage European firms against US and Chinese competitors, while the European Commission defends the rules as necessary for consumer protection and fundamental rights.

[guardian_ai_jobs.txt]
  The International Monetary Fund warns that u

In [14]:
# ── .stream() — receive tokens as they are generated ──────────────────────────
print("-" * 55)
for chunk in llm.stream("Write a 3-bullet briefing on why AI regulation matters for enterprise clients."):
    print(chunk.content, end="", flush=True)
print("\n" + "-" * 55)

-------------------------------------------------------
Here is a 3-bullet briefing on why AI regulation matters for enterprise clients:

- **Mitigating Legal & Financial Liability:** Unregulated AI systems risk violating emerging laws (e.g., EU AI Act, GDPR, sector-specific rules) on bias, privacy, and transparency. For enterprises, non-compliance can result in severe fines, litigation costs, and voided contracts, making a robust governance framework a critical tool for managing exposure.

- **Protecting Brand Trust & Customer Relationships:** Enterprise clients depend on long-term trust. An AI system that produces biased hiring decisions, hallucinates financial advice, or mishandles customer data can cause immediate reputational damage. Regulation provides a baseline for safe deployment, ensuring AI enhances brand equity rather than eroding it.

- **Enabling Scalable & Sustainable Innovation:** Regulatory compliance creates a "safe harbor" for investment. By establishing clear standa

---
## Part 2 — Document Loaders

LangChain normalises every source into the same `Document` object:

```python
Document(
    page_content = "The actual text...",
    metadata     = {"source": "reuters.txt", "page": 0, ...}
)
```

Every downstream component works on `List[Document]` regardless of source.

In [15]:
from langchain_core.documents import Document

# Convert our sample dict to Documents — identical to what any loader returns
raw_documents = [
    Document(
        page_content=text.strip(),
        metadata={"source": filename, "type": "news_article"}
    )
    for filename, text in SAMPLE_ARTICLES.items()
]

print(f"Documents: {len(raw_documents)}")
print(f"Example source  : {raw_documents[0].metadata['source']}")
print(f"Example chars   : {len(raw_documents[0].page_content)}")
print(f"Example preview : {raw_documents[0].page_content[:120]}...")

Documents: 5
Example source  : techcrunch_ai_funding.txt
Example chars   : 976
Example preview : AI Startup NeuralBridge Raises $120M Series C to Expand Enterprise Platform
TechCrunch | January 14, 2025

NeuralBridge,...


In [16]:
# ── Real file loaders (reference — uncomment as needed) ───────────────────────

# Text / Markdown
# from langchain_community.document_loaders import TextLoader
# docs = TextLoader("article.txt", encoding="utf-8").load()

# PDF — one Document per page, metadata includes page number
# from langchain_community.document_loaders import PyPDFLoader
# docs = PyPDFLoader("report.pdf").load()

# Web page / news article
# from langchain_community.document_loaders import WebBaseLoader
# docs = WebBaseLoader("https://techcrunch.com/some-article").load()
# docs = WebBaseLoader(["https://url1", "https://url2"]).load()  # multiple URLs

# CSV — one Document per row
# from langchain_community.document_loaders import CSVLoader
# docs = CSVLoader("media_mentions.csv").load()

# Whole directory
# from langchain_community.document_loaders import DirectoryLoader
# docs = DirectoryLoader("./articles", glob="**/*.txt").load()

print("Loader reference — uncomment whichever you need")

Loader reference — uncomment whichever you need


---
## Part 3 — Text Splitting

Why split at all?
1. LLM context limits — you cannot inject a 50-page report into one prompt
2. Retrieval precision — return the *relevant paragraph*, not the whole document
3. Embedding quality — short, focused passages embed more meaningfully

The overlap prevents context being lost at chunk boundaries:
```
  chunk 0: [.......................................]
  chunk 1:                       [......................................]
                                 ^--- overlap ---^
```

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter is the standard choice.
# Tries to split on \n\n first, then \n, then '. ', then ' ', then characters.
# This preserves paragraph and sentence boundaries as much as possible.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # max characters per chunk
    chunk_overlap=100,    # characters shared between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(raw_documents)

print(f"Raw documents : {len(raw_documents)}")
print(f"Chunks created: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"\nExample chunk:")
print(chunks[2].page_content)
print(f"Metadata: {chunks[2].metadata}")

Raw documents : 5
Chunks created: 11
Avg chunk size: 440 chars

Example chunk:
The funding values NeuralBridge at approximately $850 million.
The AI infrastructure market is projected to reach $200 billion by 2027, per Gartner.
Metadata: {'source': 'techcrunch_ai_funding.txt', 'type': 'news_article'}


In [18]:
# ── Other splitters (reference) ──────────────────────────────────────────────

# Token-based — precise token budget control (requires tiktoken)
# from langchain_text_splitters import TokenTextSplitter
# splitter = TokenTextSplitter(chunk_size=200, chunk_overlap=20)

# Markdown-aware — preserves headers as chunk boundaries
# from langchain_text_splitters import MarkdownTextSplitter
# splitter = MarkdownTextSplitter(chunk_size=600)

# Verify overlap between consecutive chunks
print("End of chunk 0  :", repr(chunks[0].page_content[-80:]))
print("Start of chunk 1:", repr(chunks[1].page_content[:80]))

End of chunk 0  : 'equoia Capital,\nwith participation from Google Ventures and Andreessen Horowitz.'
Start of chunk 1: 'The company, founded in 2021 by former DeepMind researchers Dr Sarah Chen and\nMa'


---
## Part 4 — Embeddings & FAISS Vector Store

An embedding maps text to a fixed-length vector of floats.
Texts with similar meaning end up close together in vector space.
This enables **semantic search** — finding passages about the same thing
even when they share no keywords with the query.

```
  "company raises funding"   -> [0.12, -0.34, 0.89, ...]  <-+ close together
  "startup secures capital"  -> [0.11, -0.31, 0.91, ...]  <-+
  "pizza recipe"             -> [-0.4,  0.67, -0.2, ...]     <- far away
```

FAISS stores these vectors in memory and finds nearest neighbours in milliseconds.

In [19]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# import numpy as np

# embeddings = GoogleGenerativeAIEmbeddings(
#     model=EMBED_MODEL,
#     google_api_key=GOOGLE_API_KEY,
# )

# # Alternative embeddings:
# # from langchain_openai import OpenAIEmbeddings
# # embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
# # Cost: $0.00002 per 1K tokens — cheap but not free like Gemini
# #
# # from langchain_community.embeddings import HuggingFaceEmbeddings
# # embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# # Runs locally — completely free but requires ~100MB download

# # Prove that semantically similar text has similar vectors
# v1 = embeddings.embed_query("AI startup raises funding round")
# v2 = embeddings.embed_query("tech company secures venture capital")
# v3 = embeddings.embed_query("pizza recipe with mozzarella")

# def cosine_sim(a, b):
#     a, b = np.array(a), np.array(b)
#     return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# print(f"Vector dimensions         : {len(v1)}")
# print(f"funding <-> venture capital: {cosine_sim(v1, v2):.4f}  (HIGH — same topic)")
# print(f"funding <-> pizza          : {cosine_sim(v1, v3):.4f}  (LOW  — different topic)")

In [20]:
import requests

# Try the two most common URL variations for 2026
urls_to_test = [
    "https://api.deepseek.com/embeddings",
    "https://api.deepseek.com/v1/embeddings"
]

for url in urls_to_test:
    print(f"Testing URL: {url}")
    headers = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}
    # Try both the V1 and the new V4-style model name
    for model in ["deepseek-embed-v1", "deepseek-v4-embed"]:
        payload = {"model": model, "input": "test"}
        res = requests.post(url, headers=headers, json=payload)
        print(f"  Model {model} -> Status: {res.status_code}")
        if res.status_code == 200:
            print(f"  ✅ SUCCESS! Use this URL/Model combo.")
            break

Testing URL: https://api.deepseek.com/embeddings
  Model deepseek-embed-v1 -> Status: 404
  Model deepseek-v4-embed -> Status: 404
Testing URL: https://api.deepseek.com/v1/embeddings
  Model deepseek-embed-v1 -> Status: 404
  Model deepseek-v4-embed -> Status: 404


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

# 'all-MiniLM-L6-v2' is fast and lightweight
# 'all-mpnet-base-v2' is slightly slower but more accurate
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'} # Change to 'cuda' if you have a GPU
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# Test data
v1 = embeddings.embed_query("AI startup raises funding round")
v2 = embeddings.embed_query("tech company secures venture capital")
v3 = embeddings.embed_query("pizza recipe with mozzarella")

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Vector dimensions         : {len(v1)}")
print(f"funding <-> venture capital: {cosine_sim(v1, v2):.4f}  (Should be HIGH)")
print(f"funding <-> pizza          : {cosine_sim(v1, v3):.4f}  (Should be LOW)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector dimensions         : 384
funding <-> venture capital: 0.4042  (Should be HIGH)
funding <-> pizza          : -0.0716  (Should be LOW)


In [22]:
from langchain_community.vectorstores import FAISS

print(f"Embedding {len(chunks)} chunks with Huggingface Embedding Models...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector store built")

# Save to disk — reload next session without re-embedding
vectorstore.save_local("media_vectorstore")
print("Saved to ./media_vectorstore/")

# To reload:
# vectorstore = FAISS.load_local("media_vectorstore", embeddings, allow_dangerous_deserialization=True)

Embedding 11 chunks with Huggingface Embedding Models...
Vector store built
Saved to ./media_vectorstore/


In [23]:
# ── Similarity search ──────────────────────────────────────────────────────────
results = vectorstore.similarity_search("Which company raised funding and how much?", k=2)
for i, doc in enumerate(results, 1):
    print(f"[{i}] {doc.metadata['source']}")
    print(f"     {doc.page_content[:200]}\n")

[1] techcrunch_ai_funding.txt
     AI Startup NeuralBridge Raises $120M Series C to Expand Enterprise Platform
TechCrunch | January 14, 2025

NeuralBridge, the San Francisco-based AI infrastructure startup, announced today
that it has 

[2] techcrunch_ai_funding.txt
     The funding values NeuralBridge at approximately $850 million.
The AI infrastructure market is projected to reach $200 billion by 2027, per Gartner.



In [24]:
# ── Search with relevance scores ──────────────────────────────────────────────
results_scored = vectorstore.similarity_search_with_relevance_scores(
    "EU AI Act fines and enforcement", k=3
)
print("Results with relevance scores (higher = more relevant):")
for doc, score in results_scored:
    print(f"  {score:.3f}  {doc.metadata['source']}  --  {doc.page_content[:100]}...")

Results with relevance scores (higher = more relevant):
  0.698  reuters_ai_regulation.txt  --  EU Regulators Tighten AI Oversight Rules, Threatening Billions in Fines
Reuters | January 16, 2025

...
  0.384  reuters_ai_regulation.txt  --  Industry groups expressed concern that the compliance burden could disadvantage
European AI firms ag...
  0.227  ft_ai_ethics.txt  --  Big Tech AI Ethics Commitments Under Scrutiny After Chatbot Failures
Financial Times | January 22, 2...


In [25]:
# ── Retriever variants ────────────────────────────────────────────────────────

# Standard similarity retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# MMR — Maximal Marginal Relevance — reduces redundancy in results
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.7}
    # lambda_mult: 0 = max diversity, 1 = max relevance
)

test = retriever.invoke("jobs automation workforce impact")
print(f"Retriever returned {len(test)} chunks:")
for doc in test:
    print(f"  * {doc.metadata['source']}: {doc.page_content[:80]}...")

Retriever returned 4 chunks:
  * guardian_ai_jobs.txt: One in Four Jobs Could Be Automated by AI Within a Decade, IMF Warns
The Guardia...
  * guardian_ai_jobs.txt: The IMF's report distinguishes between jobs that will be replaced by AI and thos...
  * ft_ai_ethics.txt: Big Tech AI Ethics Commitments Under Scrutiny After Chatbot Failures
Financial T...
  * reuters_ai_regulation.txt: Industry groups expressed concern that the compliance burden could disadvantage
...


---
## Part 5 — RAG Pipeline (Retrieval-Augmented Generation)

RAG solves the fundamental LLM problem: the model was trained months ago
and knows nothing about your client's articles.
We fix this by retrieving relevant passages at query time and injecting them
into the prompt as context.

```
  Client asks: "What did the FT say about chatbot failures?"
                        |
                [Retriever] -- finds top-k chunks from FAISS
                        |
                [PROMPT]
                  "Answer using only the context below.
                   Context: [chunk1] [chunk2] [chunk3]
                   Question: What did the FT say about..."
                        |
                [Gemini] -- reads context, generates grounded answer
                        |
                "According to ft_ai_ethics.txt, an AI chatbot
                 fabricated refund policies at an airline..."
```

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a media intelligence analyst. Answer the question using ONLY the
context provided. Cite the source filename when relevant.
If the answer is not in the context, say "Not found in the provided articles."

Context:
{context}

Question: {question}

Answer:
""")

def format_context(docs):
    parts = [f"[{doc.metadata['source']}]\n{doc.page_content}" for doc in docs]
    return "\n\n---\n\n".join(parts)

rag_chain = (
    {
        "context":  retriever | RunnableLambda(format_context),
        "question": RunnablePassthrough(),
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain ready")

RAG chain ready


In [27]:
questions = [
    "Who founded NeuralBridge and what is their background?",
    "What percentage of jobs does the IMF say are exposed to AI automation?",
    "What specific AI failures are described in the FT article?",
]

for q in questions:
    print(f"\nQ: {q}")
    print("-" * 60)
    print(rag_chain.invoke(q))


Q: Who founded NeuralBridge and what is their background?
------------------------------------------------------------
NeuralBridge was founded by former DeepMind researchers Dr Sarah Chen and Marcus Webb. (Source: techcrunch_ai_funding.txt)

Q: What percentage of jobs does the IMF say are exposed to AI automation?
------------------------------------------------------------
According to the provided context, the IMF warned that approximately 40 percent of jobs globally are exposed to AI automation (source: guardian_ai_jobs.txt).

Q: What specific AI failures are described in the FT article?
------------------------------------------------------------
According to the FT article, the specific AI failures described are: an AI customer service system at a major airline fabricated refund policies, a legal research tool hallucinated case law cited in court documents, and a healthcare chatbot provided dangerously incorrect dosage information to patients.

Source: ft_ai_ethics.txt


In [28]:
# ── RAG with source attribution ───────────────────────────────────────────────
from langchain_core.runnables import RunnableParallel

rag_with_sources = RunnableParallel(
    answer  = rag_chain,
    sources = retriever | RunnableLambda(lambda docs: list({d.metadata["source"] for d in docs}))
)

result = rag_with_sources.invoke("What is NVIDIA's quarterly revenue and what drove it?")
print("Answer:", result["answer"])
print("\nSources used:")
for s in result["sources"]:
    print(f"  * {s}")

Answer: According to the provided articles, NVIDIA reported quarterly revenue of $35.1 billion, driven almost entirely by data centre sales which reached $30.8 billion (bloomberg_nvidia.txt).

Sources used:
  * bloomberg_nvidia.txt
  * techcrunch_ai_funding.txt


In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
# CHANGE: Import from langchain_classic
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# These stay the same (core interfaces)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# --- 1. History-Aware Retriever ---
rephrase_system_prompt = (
    "Given a chat history and the latest user question "
    "formulate a standalone question which can be understood "
    "without the chat history."
)
rephrase_prompt = ChatPromptTemplate.from_messages([
    ("system", rephrase_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Create the rephrasing retriever
history_aware_retriever = create_history_aware_retriever(llm, retriever, rephrase_prompt)

# --- 2. QA Chain ---
qa_system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question."
    "\n\n"
    "{context}"
)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# --- 3. Manual Memory Loop ---
chat_history = []
turns = [
    "Tell me about the EU AI Act enforcement announced in January 2025.",
    "What fine percentage did you just mention?",
    "Which industry groups opposed these rules?",
]

for i, q in enumerate(turns, 1):
    res = rag_chain.invoke({"input": q, "chat_history": chat_history})

    chat_history.extend([
        HumanMessage(content=q),
        AIMessage(content=res["answer"]),
    ])

    if len(chat_history) > 10: chat_history = chat_history[-10:]

    srcs = {d.metadata.get("source", "Unknown") for d in res.get("context", [])}

    print(f"[Turn {i}] {q}")
    print(f"          {res['answer']}")
    if srcs:
        print(f"          Sources: {', '.join(srcs)}")
    print("-" * 20)

[Turn 1] Tell me about the EU AI Act enforcement announced in January 2025.
          The EU AI Act enforcement guidelines announced in January 2025 impose strict new rules on high-risk AI systems. Key points include:

- **Fines**: Violations can lead to penalties of up to **7% of global annual revenue**.
- **High-risk categories**: Biometric surveillance, AI used in hiring, and credit-scoring algorithms are explicitly classified as high-risk.
- **Mandatory audits**: These systems require **third-party audits** before deployment.
- **Industry reaction**: Groups like TechEurope warned the rules could disadvantage European AI firms against US and Chinese competitors.
- **EU defense**: Commissioner Helena Vasquez stated consumer protection and fundamental rights take precedence, aiming for trustworthy, transparent, and accountable AI.
          Sources: guardian_ai_jobs.txt, reuters_ai_regulation.txt, ft_ai_ethics.txt
--------------------
[Turn 2] What fine percentage did you just mention

---
## Part 6 — Media Intelligence Pipeline: Sentiment & Entity Analysis

The client deliverable: a structured JSON analysis of every article.

```json
{
  "headline":        "AI Startup NeuralBridge Raises $120M...",
  "outlet":          "TechCrunch",
  "sentiment":       "positive",
  "sentiment_score": 0.82,
  "key_topics":      ["funding", "AI infrastructure", "Series C"],
  "people":          ["Dr Sarah Chen", "Marcus Webb"],
  "organisations":   ["NeuralBridge", "Sequoia Capital"],
  "summary":         "NeuralBridge raised $120M at an $850M valuation..."
}
```

In [30]:
from langchain_core.output_parsers import JsonOutputParser

ANALYSIS_PROMPT = PromptTemplate.from_template("""
You are a media intelligence analyst. Analyse the following news article and
return a JSON object with EXACTLY these fields. No extra text, no markdown fences.

  headline        : string  (article headline)
  outlet          : string  (publication name)
  date            : string  (publication date if present, else null)
  sentiment       : string  ("positive", "negative" or "neutral")
  sentiment_score : float   (0.0 = most negative, 1.0 = most positive)
  sentiment_reason: string  (one sentence explaining the score)
  key_topics      : array   (3-5 main topics as short strings)
  people          : array   (full names of people mentioned)
  organisations   : array   (company/institution names mentioned)
  summary         : string  (2 sentences for an executive briefing)

Article:
{article}
""")

analysis_chain = ANALYSIS_PROMPT | llm | JsonOutputParser()

# Test on one article
import json
result = analysis_chain.invoke({"article": SAMPLE_ARTICLES["techcrunch_ai_funding.txt"]})
print(json.dumps(result, indent=2))

{
  "headline": "AI Startup NeuralBridge Raises $120M Series C to Expand Enterprise Platform",
  "outlet": "TechCrunch",
  "date": "January 14, 2025",
  "sentiment": "positive",
  "sentiment_score": 0.85,
  "sentiment_reason": "The article reports a substantial funding round and rapid customer growth, signaling strong market confidence and a positive outlook for the company.",
  "key_topics": [
    "AI infrastructure",
    "Series C funding",
    "enterprise expansion",
    "customer growth",
    "EU market expansion"
  ],
  "people": [
    "Dr Sarah Chen",
    "Marcus Webb"
  ],
  "organisations": [
    "NeuralBridge",
    "Sequoia Capital",
    "Google Ventures",
    "Andreessen Horowitz",
    "JPMorgan Chase",
    "Siemens",
    "UK National Health Service",
    "Gartner"
  ],
  "summary": "NeuralBridge raised $120 million in Series C funding led by Sequoia Capital to expand its enterprise AI platform. The company plans to grow its engineering team and open offices in London and Ber

In [31]:
# ── Batch process all articles in parallel ─────────────────────────────────────
batch_inputs = [{"article": text} for text in SAMPLE_ARTICLES.values()]
article_names = list(SAMPLE_ARTICLES.keys())

print("Analysing all articles in parallel...")
analyses = analysis_chain.batch(batch_inputs)

print(f"\nProcessed {len(analyses)} articles\n")
print(f"{'Article':<35} {'Sentiment':<12} {'Score':<8} Topics")
print("-" * 90)
for name, a in zip(article_names, analyses):
    topics = ", ".join(a.get("key_topics", [])[:3])
    print(f"{name:<35} {a.get('sentiment',''):<12} {a.get('sentiment_score', 0):<8.2f} {topics}")

Analysing all articles in parallel...

Processed 5 articles

Article                             Sentiment    Score    Topics
------------------------------------------------------------------------------------------
techcrunch_ai_funding.txt           positive     0.85     funding, AI infrastructure, enterprise expansion
reuters_ai_regulation.txt           negative     0.30     AI regulation, EU AI Act, fines
guardian_ai_jobs.txt                neutral      0.40     AI automation, job displacement, retraining programmes
bloomberg_nvidia.txt                positive     0.90     record quarterly revenue, AI chip demand, data center sales
ft_ai_ethics.txt                    negative     0.25     AI ethics, chatbot failures, accountability


In [32]:
# ── Client dashboard briefing ─────────────────────────────────────────────────
BRIEFING_PROMPT = PromptTemplate.from_template("""
You are preparing a morning media briefing for a senior executive.
Below are structured analyses of {n} news articles from the past week.

Analyses:
{analyses}

Write a concise briefing covering:
1. Overall sentiment trend
2. The 2-3 most important stories and why they matter
3. Risks or opportunities the executive should know about
4. Key people and organisations appearing across multiple articles

Tone: professional, factual, no more than 300 words.
""")

briefing_chain = BRIEFING_PROMPT | llm | StrOutputParser()

analyses_text = "\n\n".join(
    f"Article: {name}\nSentiment: {a.get('sentiment')} ({a.get('sentiment_score', 0):.2f})\n"
    f"Topics: {', '.join(a.get('key_topics',[]))}\nSummary: {a.get('summary', '')}"
    for name, a in zip(article_names, analyses)
)

briefing = briefing_chain.invoke({"n": len(analyses), "analyses": analyses_text})
print("=" * 60)
print("WEEKLY MEDIA INTELLIGENCE BRIEFING")
print("=" * 60)
print(briefing)

WEEKLY MEDIA INTELLIGENCE BRIEFING
**Morning Media Briefing – Week in Review**

**Overall Sentiment Trend**  
Mixed. Strong positive sentiment from AI infrastructure growth (NVIDIA, NeuralBridge) is tempered by negative regulatory and ethical headwinds (EU AI Act, chatbot failures) and neutral long-term workforce concerns (IMF).

**Key Stories & Why They Matter**  
1. **NVIDIA’s Record Revenue ($35.1B)** – Exceeded expectations by $2.1B, driven by insatiable AI chip demand. CEO Jensen Huang confirmed supply still lags demand for the new Blackwell GPU. **Relevance:** Confirms booming AI infrastructure spending but flags concentration risk (top 4 customers – Microsoft, Google, Amazon, Meta).  
2. **EU AI Act Enforcement Guidelines** – Fines up to 7% of global revenue for high-risk systems. Industry warns of compliance burden harming European innovation. **Relevance:** Direct impact on any EU market expansion; creates legal and cost risks.  
3. **IMF Jobs Warning** – Up to 40% of global j

In [33]:
# ── Pydantic output parser — typed, validated structured output ───────────────
# More robust than JsonOutputParser: validates field types automatically

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

class ArticleAnalysis(BaseModel):
    headline:         str        = Field(description="Article headline")
    sentiment:        str        = Field(description="positive, negative or neutral")
    sentiment_score:  float      = Field(description="0.0 to 1.0")
    key_topics:       List[str]  = Field(description="3-5 main topics")
    people:           List[str]  = Field(description="People mentioned")
    organisations:    List[str]  = Field(description="Organisations mentioned")
    summary:          str        = Field(description="2-sentence executive summary")

pydantic_parser = PydanticOutputParser(pydantic_object=ArticleAnalysis)

typed_prompt = PromptTemplate(
    template="Analyse this news article and return structured output.\n{format_instructions}\n\nArticle:\n{article}",
    input_variables=["article"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()},
)

typed_chain = typed_prompt | llm | pydantic_parser

typed_result = typed_chain.invoke({"article": SAMPLE_ARTICLES["reuters_ai_regulation.txt"]})

print(type(typed_result))       # ArticleAnalysis — a real Python object
print(f"Headline  : {typed_result.headline}")
print(f"Sentiment : {typed_result.sentiment} ({typed_result.sentiment_score})")
print(f"People    : {typed_result.people}")
print(f"Orgs      : {typed_result.organisations}")
print(f"Summary   : {typed_result.summary}")

<class '__main__.ArticleAnalysis'>
Headline  : EU Regulators Tighten AI Oversight Rules, Threatening Billions in Fines
Sentiment : negative (0.3)
People    : ['Helena Vasquez']
Orgs      : ['European Union', 'European AI Office', 'European Commission', 'TechEurope']
Summary   : EU regulators issued new enforcement guidelines for the AI Act, classifying biometric surveillance and hiring algorithms as high-risk and subject to mandatory audits, with fines up to 7% of global revenue. Industry groups like TechEurope criticized the rules as disproportionate, while the European Commission defended them as necessary for consumer protection.


---
## Part 7 — Agentic AI: ReAct Agent with Tools

A chain has a fixed execution order. An agent is different:
it reads the question, **decides which tool to call**, observes the result,
and repeats until it can answer. This is the **ReAct loop** (Reason + Act).

```
  User: "Find the NVIDIA article, then analyse its sentiment."
            |
  THOUGHT: "I need to find the NVIDIA article first."
  ACTION:  search_articles("NVIDIA revenue earnings")
  OBSERVE: [bloomberg_nvidia.txt chunk returned]
            |
  THOUGHT: "Now I should analyse the sentiment."
  ACTION:  analyse_sentiment("[bloomberg text]")
  OBSERVE: "positive, score 0.85, driven by record earnings"
            |
  FINAL ANSWER: "The Bloomberg article is positive (0.85).
                 Record $35B revenue, 122% YoY growth..."
```

The agent never follows a hard-coded path — it adapts to each question.

In [34]:
from langchain_core.tools import tool

# The @tool decorator turns a Python function into an agent tool.
# The docstring IS the description the agent reads to decide when to use it.
# Write docstrings as instructions to the agent, not to a human developer.

@tool
def search_articles(query: str) -> str:
    """
    Search the media article archive for content relevant to the query.
    Use this whenever the user asks about something that might be in the news articles.
    Returns the most relevant passages with their source filenames.
    Input: a natural language search query.
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant articles found."
    parts = [f"[{doc.metadata['source']}]\n{doc.page_content.strip()}" for doc in docs]
    return "\n\n---\n\n".join(parts)


@tool
def analyse_sentiment(text: str) -> str:
    """
    Analyse the sentiment of a given piece of text.
    Returns sentiment label (positive/negative/neutral), a score 0-1, and a one-line reason.
    Use this when asked to evaluate tone or sentiment of any text or article.
    Input: the text to analyse.
    """
    prompt = (
        'Return ONLY a JSON object: {"sentiment": "positive|negative|neutral", '
        '"score": 0.0-1.0, "reason": "one sentence"}\n\n' + text
    )
    return llm.invoke(prompt).content


@tool
def extract_entities(text: str) -> str:
    """
    Extract named entities from text: people, organisations, places and dates.
    Use this when asked to identify who or what is mentioned in an article.
    Input: the text to analyse.
    """
    prompt = (
        'Extract named entities. Return ONLY JSON: '
        '{"people": [...], "organisations": [...], "places": [...], "dates": [...]}\n\n' + text
    )
    return llm.invoke(prompt).content


@tool
def summarise_text(text: str) -> str:
    """
    Summarise a piece of text in 2-3 bullet points.
    Use this when asked to condense or give a brief overview of any content.
    Input: the text to summarise.
    """
    return llm.invoke(f"Summarise in 2-3 bullet points:\n\n{text}").content


@tool
def count_articles_by_sentiment(sentiment: str) -> str:
    """
    Count how many of the loaded articles match a given sentiment label.
    Use when asked how many positive, negative or neutral articles exist.
    Input: 'positive', 'negative', or 'neutral'.
    """
    count = 0
    for text in SAMPLE_ARTICLES.values():
        resp = llm.invoke(f"Is this article primarily {sentiment}? Answer only yes or no.\n\n{text[:400]}").content.strip().lower()
        if "yes" in resp:
            count += 1
    return f"{count} of {len(SAMPLE_ARTICLES)} articles are classified as {sentiment}."


all_tools = [search_articles, analyse_sentiment, extract_entities, summarise_text, count_articles_by_sentiment]
print(f"Defined {len(all_tools)} tools:")
for t in all_tools:
    print(f"  * {t.name}: {t.description.strip().splitlines()[0]}")

Defined 5 tools:
  * search_articles: Search the media article archive for content relevant to the query.
  * analyse_sentiment: Analyse the sentiment of a given piece of text.
  * extract_entities: Extract named entities from text: people, organisations, places and dates.
  * summarise_text: Summarise a piece of text in 2-3 bullet points.
  * count_articles_by_sentiment: Count how many of the loaded articles match a given sentiment label.


In [35]:
# 1. NEW IMPORT PATH (2026 Standard)
from langchain_classic import hub

# 2. These remain in classic for AgentExecutor support
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic.memory import ConversationBufferWindowMemory

# 3. Rest of your code remains the same
react_prompt = hub.pull("hwchase17/react")

agent_memory = ConversationBufferWindowMemory(
    k=6,
    memory_key="chat_history",
    return_messages=True,
)

# create_react_agent (ensure llm and all_tools are already defined)
agent = create_react_agent(llm=llm, tools=all_tools, prompt=react_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=all_tools,
    memory=agent_memory,
    verbose=True,
    max_iterations=8,
    handle_parsing_errors=True,
)

print(f"Agent ready with {len(all_tools)} tools")

Agent ready with 5 tools


/tmp/ipykernel_3113/728422738.py:9: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  react_prompt = hub.pull("hwchase17/react")
/tmp/ipykernel_3113/728422738.py:11: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  agent_memory = ConversationBufferWindowMemory(


In [36]:
# Single tool call
result = agent_executor.invoke({"input": "What funding did NeuralBridge raise and who led the round?"})
print("\nFINAL ANSWER:", result["output"])



> Entering new AgentExecutor chain...
Thought: The user is asking about a specific company's funding round. I should search the article archive for relevant news about NeuralBridge.

Action: search_articles
Action Input: "NeuralBridge funding led round"[techcrunch_ai_funding.txt]
AI Startup NeuralBridge Raises $120M Series C to Expand Enterprise Platform
TechCrunch | January 14, 2025

NeuralBridge, the San Francisco-based AI infrastructure startup, announced today
that it has closed a $120 million Series C funding round led by Sequoia Capital,
with participation from Google Ventures and Andreessen Horowitz.

---

[techcrunch_ai_funding.txt]
The funding values NeuralBridge at approximately $850 million.
The AI infrastructure market is projected to reach $200 billion by 2027, per Gartner.

---

[techcrunch_ai_funding.txt]
The company, founded in 2021 by former DeepMind researchers Dr Sarah Chen and
Marcus Webb, has grown its enterprise customer base from 40 to over 300 companies
in the

In [37]:
# Multi-step: search THEN sentiment analysis
result = agent_executor.invoke({
    "input": "Find the NVIDIA article, analyse its sentiment, and tell me if investors should be optimistic."
})
print("\nFINAL ANSWER:", result["output"])



> Entering new AgentExecutor chain...
Thought: I need to find an article about NVIDIA first. I'll search for "NVIDIA" in the article archive.

Action: search_articles
Action Input: "NVIDIA"[bloomberg_nvidia.txt]
NVIDIA Posts Record $35B Quarter as AI Chip Demand Shows No Sign of Slowing
Bloomberg | January 20, 2025

NVIDIA reported quarterly revenue of $35.1 billion on Wednesday, smashing analyst
expectations of $33 billion and representing a 122 percent year-over-year increase.
The results were driven almost entirely by data centre sales, which reached $30.8
billion as hyperscalers and enterprise customers continued to build out AI infrastructure.

---

[bloomberg_nvidia.txt]
CEO Jensen Huang told analysts that demand for the company's Blackwell GPU architecture
continues to exceed supply. Shares rose 8 percent in after-hours trading.

Analysts at Morgan Stanley raised their price target for NVIDIA to $900 per share.
However, some investors have raised concerns about concentration r

In [38]:
# Multi-step: search THEN entity extraction
result = agent_executor.invoke({
    "input": "Search for articles about EU regulation, then extract all organisations and people mentioned."
})
print("\nFINAL ANSWER:", result["output"])



> Entering new AgentExecutor chain...
Thought: I need to find articles about EU regulation to extract organizations and people mentioned. I'll start by searching for relevant articles.

Action: search_articles
Action Input: "EU regulation"[reuters_ai_regulation.txt]
EU Regulators Tighten AI Oversight Rules, Threatening Billions in Fines
Reuters | January 16, 2025

European Union regulators announced sweeping new enforcement guidelines for the
EU AI Act on Thursday, warning that violations could result in fines of up to
7 percent of global annual revenue for companies deploying high-risk AI systems.

The guidelines, issued by the European AI Office in Brussels, clarify that biometric
surveillance systems, AI used in hiring decisions, and credit-scoring algorithms
will all be classified as high-risk and subject to mandatory third-party audits.

---

[reuters_ai_regulation.txt]
Industry groups expressed concern that the compliance burden could disadvantage
European AI firms against US a

In [39]:
# Memory test — follow-up question
agent_executor.invoke({"input": "What does the IMF report say about job automation?"})
result = agent_executor.invoke({"input": "Which countries did you just mention?"})
print("\nFINAL ANSWER (uses memory):", result["output"])



> Entering new AgentExecutor chain...
Thought: The user is asking about what an IMF report says regarding job automation. I need to search for relevant articles from the media archive to find the report's content.

Action: search_articles
Action Input: "IMF report job automation"[guardian_ai_jobs.txt]
The IMF's report distinguishes between jobs that will be replaced by AI and those
that will be augmented. Routine cognitive tasks — data entry, basic analysis,
customer service — face the greatest risk. Creative, social and physical roles
are considered more resilient.

The fund called on governments to invest urgently in retraining programmes and
to reform tax systems that currently favour capital over labour. IMF Managing
Director Kristalina Georgieva said the transition could either increase inequality
dramatically or create a more prosperous society depending on policy choices.

---

[guardian_ai_jobs.txt]
One in Four Jobs Could Be Automated by AI Within a Decade, IMF Warns
The Guar

---
## Part 8 — Common LangChain Patterns Reference

All patterns you will encounter in real projects.

In [40]:
# ── RunnableParallel — run multiple chains at once, merge results ─────────────
from langchain_core.runnables import RunnableParallel, RunnableLambda

parallel_chain = RunnableParallel(
    sentiment  = (PromptTemplate.from_template("Sentiment of this text? One word.\n\n{text}") | llm | StrOutputParser()),
    summary    = (PromptTemplate.from_template("One sentence summary:\n\n{text}") | llm | StrOutputParser()),
    word_count = RunnableLambda(lambda x: f"{len(x['text'].split())} words"),
)

parallel_result = parallel_chain.invoke({"text": SAMPLE_ARTICLES["bloomberg_nvidia.txt"]})
for k, v in parallel_result.items():
    print(f"  {k:12}: {v}")

  sentiment   : Positive
  summary     : NVIDIA reported a record $35.1 billion quarterly revenue, beating estimates and driven by surging AI chip demand, though investor concerns persist over concentration risk from major customers like Microsoft and Google.
  word_count  : 128 words


In [41]:
# ── RunnableBranch — conditional routing ──────────────────────────────────────
from langchain_core.runnables import RunnableBranch

positive_chain = (PromptTemplate.from_template("Write a 1-sentence opportunity note for this positive coverage:\n{text}") | llm | StrOutputParser())
negative_chain = (PromptTemplate.from_template("Write a 1-sentence risk alert for this negative coverage:\n{text}") | llm | StrOutputParser())
neutral_chain  = (PromptTemplate.from_template("Write a 1-sentence factual note for this neutral coverage:\n{text}") | llm | StrOutputParser())

routed_chain = RunnableBranch(
    (lambda x: "positive" in x["sentiment"].lower(), positive_chain),
    (lambda x: "negative" in x["sentiment"].lower(), negative_chain),
    neutral_chain,  # default
)

r1 = routed_chain.invoke({"sentiment": "positive", "text": SAMPLE_ARTICLES["bloomberg_nvidia.txt"]})
r2 = routed_chain.invoke({"sentiment": "negative", "text": SAMPLE_ARTICLES["guardian_ai_jobs.txt"]})
print("Positive route:", r1)
print("Negative route:", r2)

Positive route: NVIDIA's record $35B quarter and 122% revenue surge, driven by insatiable AI chip demand with Blackwell supply still trailing, present a compelling opportunity to capitalize on the company's dominant position in the AI infrastructure buildout.
Negative route: IMF warns that up to 60% of jobs in advanced economies face AI-driven disruption within a decade, risking mass unemployment and soaring inequality without urgent government action.


In [42]:
# ── .with_retry() — automatic retry on API failures ──────────────────────────
resilient_llm = llm.with_retry(
    retry_if_exception_type=(Exception,),
    stop_after_attempt=3,
    wait_exponential_jitter=True,
)
result = (PromptTemplate.from_template("What is 2+2? Just the number.") | resilient_llm | StrOutputParser()).invoke({})
print("Resilient chain:", result)

Resilient chain: 4


In [43]:
# ── .with_fallbacks() — try one model, fall back to another on failure ──────────
# from langchain_openai import ChatOpenAI
# backup = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
# safe_llm = llm.with_fallbacks([backup])
# Now if Gemini fails (rate limit, outage), it automatically tries GPT-4o-mini

safe_llm = llm.with_fallbacks([llm])  # demo: falls back to itself
print(safe_llm.invoke("Say 'fallback working' in 3 words.").content)

fallback is working


In [44]:
# ── Memory types (2026 Updated Imports) ───────────────────────────────────────────
from langchain_classic.memory import (
    ConversationBufferMemory,        # Keeps ALL history
    ConversationBufferWindowMemory,  # Keeps last k turns
    ConversationSummaryMemory        # Summarizes old history
)

# 1. Demonstrate ConversationBufferWindowMemory
# k=3 means it will only keep the 3 most recent interaction pairs.
mem = ConversationBufferWindowMemory(k=3, return_messages=True)

# Simulate interactions
mem.save_context({"input": "Hello"}, {"output": "Hi there!"})
mem.save_context({"input": "What is RAG?"}, {"output": "Retrieval-Augmented Generation"})
mem.save_context({"input": "Is it useful for AML?"}, {"output": "Yes, for financial crime compliance."})

print("--- Window Memory Buffer ---")
print(mem.load_memory_variables({}))

# 2. Example: ConversationSummaryMemory (Requires LLM)
# Note: You must have your 'llm' object (e.g., DeepSeek) defined first.
summary_mem = ConversationSummaryMemory(llm=llm, return_messages=True)
summary_mem.save_context({"input": "Explain KYC."}, {"output": "Know Your Customer is a verification process."})

print("\n--- Summary Memory Buffer ---")
print(summary_mem.load_memory_variables({}))

--- Window Memory Buffer ---
{'history': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is RAG?', additional_kwargs={}, response_metadata={}), AIMessage(content='Retrieval-Augmented Generation', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Is it useful for AML?', additional_kwargs={}, response_metadata={}), AIMessage(content='Yes, for financial crime compliance.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


/tmp/ipykernel_3113/3538213980.py:22: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  summary_mem = ConversationSummaryMemory(llm=llm, return_messages=True)



--- Summary Memory Buffer ---
{'history': [SystemMessage(content='The human asks the AI to explain KYC, and the AI responds that Know Your Customer is a verification process.', additional_kwargs={}, response_metadata={})]}


In [45]:
# ── Custom output parser ─────────────────────────────────────────────────────
from langchain_core.output_parsers import BaseOutputParser
from typing import List

class BulletListParser(BaseOutputParser):
    """Parses LLM output into a clean Python list, removing bullet characters."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().splitlines()
        return [line.strip().lstrip("*-+.0123456789 ") for line in lines if line.strip()]

bullet_chain = (
    PromptTemplate.from_template("List the 4 main risks in this article as bullet points:\n\n{article}")
    | llm
    | BulletListParser()
)

risks = bullet_chain.invoke({"article": SAMPLE_ARTICLES["reuters_ai_regulation.txt"]})
print(type(risks))    # list!
for i, risk in enumerate(risks, 1):
    print(f"  {i}. {risk}")

<class 'list'>
  1. Fines of up to 7% of global annual revenue for violations involving high-risk AI systems
  2. Competitive disadvantage for European AI firms against US and Chinese rivals due to compliance burden
  3. Potential damage to European innovation, as described by industry groups as "disproportionate"
  4. Mandatory third-party audits for high-risk AI systems such as biometric surveillance, hiring tools, and credit-scoring algorithms


In [46]:
# ── Async execution — non-blocking calls for web apps ────────────────────────
import asyncio

async def analyse_async(article_text: str) -> str:
    chain = (
        PromptTemplate.from_template("What is the main risk in this article? One sentence.\n\n{article}")
        | llm
        | StrOutputParser()
    )
    return await chain.ainvoke({"article": article_text})

async def run_all_async():
    tasks = [analyse_async(text) for text in SAMPLE_ARTICLES.values()]
    return await asyncio.gather(*tasks)   # all run concurrently

async_results = await run_all_async()

print("Async results (all ran concurrently):")
for name, result in zip(SAMPLE_ARTICLES.keys(), async_results):
    print(f"  [{name}] {result}")

Async results (all ran concurrently):
  [techcrunch_ai_funding.txt] The main risk is that NeuralBridge's rapid expansion—tripling its engineering team and opening new offices in Europe—could outpace its ability to maintain operational focus, product quality, and regulatory compliance in a highly competitive AI infrastructure market.
  [reuters_ai_regulation.txt] The main risk highlighted in the article is that the compliance burden from the EU's new AI oversight rules could disadvantage European AI firms against US and Chinese competitors.
  [guardian_ai_jobs.txt] The main risk is that AI automation could displace up to 40% of jobs globally, particularly routine cognitive tasks, potentially exacerbating inequality if governments fail to implement retraining programs and tax reforms.
  [bloomberg_nvidia.txt] The main risk is that NVIDIA's revenue is heavily concentrated among just four customers—Microsoft, Google, Amazon, and Meta—which together account for about 40% of total sales.
  [

In [47]:
# ── Inspect a chain's computation graph ───────────────────────────────────────
graph = rag_chain.get_graph()
print("RAG chain nodes:")
for node_id, node in graph.nodes.items():
    print(f"  {node.name}")

RAG chain nodes:
  Parallel<context>Input
  Parallel<context>Output
  Branch
  Passthrough
  Parallel<answer>Input
  Parallel<answer>Output
  Parallel<context>Input
  Parallel<context>Output
  PromptTemplate
  Passthrough
  ChatPromptTemplate
  ChatOpenAI
  StrOutputParser
  Passthrough


---
## Part 9 — Full Client Demo

Everything wired together into a single client session.

In [48]:
# Fresh agent with clean memory for the demo
demo_memory = ConversationBufferWindowMemory(k=6, memory_key="chat_history", return_messages=True)
demo_agent  = AgentExecutor(
    agent=agent, tools=all_tools, memory=demo_memory,
    verbose=False, max_iterations=8, handle_parsing_errors=True,
)

client_questions = [
    "Give me a one-line summary of each article in the archive.",
    "Which article has the most negative tone and why?",
    "What do the articles say about regulatory risk for AI companies?",
    "Extract all financial figures mentioned across the articles.",
]

print("CLIENT SESSION")
print("=" * 60)
for q in client_questions:
    print(f"\nClient: {q}")
    result = demo_agent.invoke({"input": q})
    print(f"Agent : {result['output']}")

CLIENT SESSION

Client: Give me a one-line summary of each article in the archive.
Agent : **  
Based on the retrieved articles from the archive (three of five articles found via search), here are one‑line summaries:

- **ft_ai_ethics.txt**: AI ethics article highlights the gap between companies’ published safety principles and actual testing, with concerns over “ethics washing”.  
- **reuters_ai_regulation.txt**: EU regulators announce stricter AI oversight rules, threatening fines of up to 7% of global revenue for high‑risk AI systems.  
- **techcrunch_ai_funding.txt**: AI startup founded by former DeepMind researchers secures funding, expands enterprise customer base, and plans to open offices in London and Berlin.

Note: The archive contains five articles according to sentiment counts, but only three were retrievable via the search tool. The remaining two articles could not be summarised due to lack of content returned.

Client: Which article has the most negative tone and why?
Age

In [49]:
# ── Interactive loop — uncomment to chat live ────────────────────────────────
# Type 'quit' to exit, 'reset' to clear memory.

# def run_interactive():
#     print("\nMedia Intelligence Agent -- type questions or 'quit' to exit\n")
#     while True:
#         try:
#             user_input = input("You: ").strip()
#         except (EOFError, KeyboardInterrupt):
#             break
#         if not user_input:
#             continue
#         if user_input.lower() in ("quit", "exit"):
#             break
#         if user_input.lower() == "reset":
#             demo_memory.clear()
#             print("Memory cleared.")
#             continue
#         result = demo_agent.invoke({"input": user_input})
#         print(f"Agent: {result['output']}\n")
#
# run_interactive()

---
## Summary

| Part | LangChain components covered |
|------|------------------------------|
| 1 — Basics | `ChatGoogleGenerativeAI`, `ChatPromptTemplate`, `PromptTemplate`, `StrOutputParser`, `JsonOutputParser`, `.batch()`, `.stream()` |
| 2 — Loaders | `Document`, `TextLoader`, `PyPDFLoader`, `WebBaseLoader`, `CSVLoader`, `DirectoryLoader` |
| 3 — Splitting | `RecursiveCharacterTextSplitter`, `TokenTextSplitter`, `MarkdownTextSplitter` |
| 4 — Embeddings | `GoogleGenerativeAIEmbeddings`, `FAISS`, `.similarity_search()`, `.similarity_search_with_relevance_scores()`, `.as_retriever()`, MMR |
| 5 — RAG | `RunnablePassthrough`, `RunnableLambda`, `RunnableParallel`, `ConversationalRetrievalChain`, `ConversationBufferWindowMemory` |
| 6 — Analysis | `PydanticOutputParser`, `BaseModel`, `.batch()` for parallel processing |
| 7 — Agent | `@tool`, `create_react_agent`, `AgentExecutor`, `hub.pull()` |
| 8 — Patterns | `RunnableParallel`, `RunnableBranch`, `.with_retry()`, `.with_fallbacks()`, `BaseOutputParser`, `ainvoke`, `.get_graph()` |

## Model swap cheat-sheet

```python
# Gemini (this notebook)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", google_api_key=KEY, convert_system_message_to_human=True)
emb = GoogleGenerativeAIEmbeddings(model="models/embedding-text-001", google_api_key=KEY)

# OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(model="gpt-4o", api_key=KEY)
emb = OpenAIEmbeddings(model="text-embedding-3-small", api_key=KEY)

# Anthropic
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-opus-4-5", api_key=KEY)
# (use OpenAI embeddings with Anthropic — Anthropic has no embedding model)
```

All chain, RAG and agent code in this notebook works unchanged with any of the above.